# 03a — EU Pipeline (2013 Sample)

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 1******6  

---

## Purpose

This notebook takes the raw EU 2013 window data from notebook 02 and produces a single **analysis-ready dataset**: one row per trading day, every variable as a column, all sources aligned to the same daily date index.

## What this notebook does — step by step

| Step | What happens |
|---|---|
| 1 | Imports, paths, constants |
| 2 | Load all raw EU2013 CSVs |
| 3 | Clean volatility surface (filter -99, standardise columns) |
| 4 | Compute smile parameters per day |
| 5 | Clean and align control variables to trading-day index |
| 6 | Merge everything into one daily panel |
| 7 | Final validation of merged dataset |
| 8 | Save to data/intermediate/ |

## Data lineage
- **Input:** `data/raw/window_eu_2013/*.csv` (read-only, never modified)
- **Output:** `data/intermediate/eu2013_analysis_ready.csv`

## Key design decisions
- Raw data is NEVER modified. All transformations produce new objects.
- Every filter logs row counts before and after.
- The trading-day index is derived from `crsp.dsi` (most reliable daily calendar).
- Smile parameters follow Malz (1997): IV at fixed delta/maturity nodes from the OptionMetrics surface.

---

> ⚠️ Run notebook 02 first to ensure raw CSVs exist in data/raw/window_eu_2013/

## Step 1 — Imports, paths, constants

In [1]:
import os
import glob
import datetime
import json
import numpy as np
import pandas as pd

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------
RAW_EU       = "../data/raw/window_eu_2013"
INTERMEDIATE = "../data/intermediate"
LOG_DIR      = "../logs"

os.makedirs(INTERMEDIATE, exist_ok=True)
os.makedirs(LOG_DIR,      exist_ok=True)

TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# ----------------------------------------------------------------
# Constants
# ----------------------------------------------------------------

# OptionMetrics sentinel value for uncomputable IV
# Source: OptionMetrics Ivy DB Europe Reference Manual
IV_SENTINEL = -99

# Security of interest in the EU sample
# From notebook 02: security_name shows securityid=500096 = Adidas AG
EU_SECURITYID = 500096.0

# Maturity nodes available in the surface (in calendar days)
# We will inspect the actual values after loading
# Tompkins (2001) uses standardised maturities — we follow what is in the data
TARGET_DAYS = [30, 60, 91, 182, 365]  # will be confirmed after load

# Delta nodes: OptionMetrics surface uses integer deltas (20, 25, 30 ... 80)
# for both calls and puts
# ATM is typically delta=50
TARGET_DELTAS = [20, 25, 50, 75, 80]  # will be confirmed after load

# Transformation log — records every row count change
transform_log = []

def log_transform(step, table, before, after, reason):
    """Record every row count change for auditability."""
    entry = {
        "step":    step,
        "table":   table,
        "before":  before,
        "after":   after,
        "dropped": before - after,
        "reason":  reason,
    }
    transform_log.append(entry)
    print(f"  [{step}] {table}: {before} → {after} rows (dropped {before - after}: {reason})")

print(f"Timestamp : {TIMESTAMP}")
print(f"RAW_EU    : {RAW_EU}")
print(f"INTERMEDIATE: {INTERMEDIATE}")
print("\nConstants set. Ready.")

Timestamp : 20260312_113334
RAW_EU    : ../data/raw/window_eu_2013
INTERMEDIATE: ../data/intermediate

Constants set. Ready.


## Step 2 — Load all raw EU2013 CSVs

We load each file and confirm the row counts match what we pulled in notebook 02.

**Expected counts (from notebook 02):**
- `volatility_surface_2013`: 2,860 rows
- `option_price_2013`: 8,740 rows
- `security_price`: 50 rows
- `historical_volatility`: 143 rows
- `security_name`: 1 row
- `frb.rates_daily`: 120 rows
- `crsp.dsi`: 82 rows
- `crsp.dsp500`: 82 rows
- `ff.factors_daily`: 82 rows
- `cboe.cboe`: 82 rows
- `comp.g_idx_daily`: 45,307 rows
- `comp.g_exrt_dly`: 20,719 rows
- `wrdsapps.eushort`: 6,695 rows

In [2]:
def load_latest(prefix, folder=RAW_EU):
    """
    Load the most recently saved CSV matching a filename prefix.
    This handles the timestamped filenames from notebook 02.
    """
    pattern = os.path.join(folder, f"{prefix}*.csv")
    matches = sorted(glob.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file found matching: {pattern}")
    path = matches[-1]  # most recent
    df = pd.read_csv(path)
    print(f"✓ Loaded {os.path.basename(path)}: {df.shape[0]} rows x {df.shape[1]} cols")
    return df

print("Loading EU2013 raw files...\n")

# Core OptionMetrics EU tables
df_surface  = load_latest("optionmsamp_europe__volatility_surface_2013")
df_optprice = load_latest("optionmsamp_europe__option_price_2013")
df_secprice = load_latest("optionmsamp_europe__security_price")
df_histvol  = load_latest("optionmsamp_europe__historical_volatility")
df_secname  = load_latest("optionmsamp_europe__security_name")

# Control tables
df_frb      = load_latest("frb__rates_daily")
df_dsi      = load_latest("crsp__dsi")
df_dsp500   = load_latest("crsp__dsp500")
df_ff       = load_latest("ff__factors_daily")
df_vix      = load_latest("cboe__cboe")
df_gidx     = load_latest("comp__g_idx_daily")
df_fx       = load_latest("comp__g_exrt_dly")
df_short    = load_latest("wrdsapps__eushort")

print("\n✓ All files loaded.")

Loading EU2013 raw files...

✓ Loaded optionmsamp_europe__volatility_surface_2013__EU2013__20260308_171256.csv: 2860 rows x 10 cols
✓ Loaded optionmsamp_europe__option_price_2013__EU2013__20260308_171256.csv: 8740 rows x 24 cols
✓ Loaded optionmsamp_europe__security_price__EU2013__20260308_171256.csv: 50 rows x 14 cols
✓ Loaded optionmsamp_europe__historical_volatility__EU2013__20260308_171256.csv: 143 rows x 5 cols
✓ Loaded optionmsamp_europe__security_name__EU2013__20260308_171256.csv: 1 rows x 6 cols
✓ Loaded frb__rates_daily__EU2013__20260308_171256.csv: 120 rows x 83 cols
✓ Loaded crsp__dsi__EU2013__20260308_171256.csv: 82 rows x 11 cols
✓ Loaded crsp__dsp500__EU2013__20260308_171256.csv: 82 rows x 11 cols
✓ Loaded ff__factors_daily__EU2013__20260308_171256.csv: 82 rows x 6 cols
✓ Loaded cboe__cboe__EU2013__20260308_171256.csv: 82 rows x 17 cols
✓ Loaded comp__g_idx_daily__EU2013__20260308_171256.csv: 45307 rows x 10 cols
✓ Loaded comp__g_exrt_dly__EU2013__20260308_171256.csv: 207

## Step 3 — Clean the volatility surface

### What we do and why

**3a. Parse dates** — date columns arrive as strings from WRDS. We convert to `datetime64` for all subsequent operations.

**3b. Filter to our security** — the sample may contain multiple securities. We keep only `securityid = 500096` (Adidas AG in our sample; will be OESX with full access).

**3c. Drop sentinel IVs (-99)** — OptionMetrics uses -99 to flag cases where Black-Scholes inversion failed (deep ITM/OTM options, arbitrage violations, illiquid strikes). These are NOT missing at random and must be excluded before any analysis. Source: OptionMetrics Ivy DB Europe Reference Manual.

**3d. Drop non-positive IVs** — any IV ≤ 0 is economically meaningless (volatility is always positive).

**3e. Inspect the delta and days grid** — we must know what nodes are actually present before constructing smile parameters.

In [3]:
print("=" * 60)
print("STEP 3 — Clean volatility surface")
print("=" * 60)

surf = df_surface.copy()  # never modify raw
n0 = len(surf)
print(f"\nStarting rows: {n0}")

# 3a. Parse dates
surf['date'] = pd.to_datetime(surf['date'])
print(f"\n[3a] Date range: {surf['date'].min().date()} → {surf['date'].max().date()}")
print(f"     Unique trading days: {surf['date'].nunique()}")

# 3b. Filter to our security
n_before = len(surf)
surf = surf[surf['securityid'] == EU_SECURITYID]
log_transform("3b", "surface", n_before, len(surf),
              f"keep securityid={EU_SECURITYID} only")

# 3c. Drop sentinel IVs (-99)
n_before = len(surf)
surf = surf[surf['impliedvol'] != IV_SENTINEL]
log_transform("3c", "surface", n_before, len(surf),
              "drop IV sentinel -99 (inversion failure)")

# 3d. Drop non-positive IVs
n_before = len(surf)
surf = surf[surf['impliedvol'] > 0]
log_transform("3d", "surface", n_before, len(surf),
              "drop IV <= 0 (economically invalid)")

# 3e. Inspect the grid
print(f"\n[3e] Delta nodes present : {sorted(surf['delta'].unique())}")
print(f"     Days nodes present   : {sorted(surf['days'].unique())}")
print(f"     Call/put flags       : {surf['callput'].unique()}")
print(f"     Unique trading days  : {surf['date'].nunique()}")
print(f"\nSurface after cleaning: {len(surf)} rows")

STEP 3 — Clean volatility surface

Starting rows: 2860

[3a] Date range: 2013-03-01 → 2013-03-15
     Unique trading days: 11
  [3b] surface: 2860 → 2860 rows (dropped 0: keep securityid=500096.0 only)
  [3c] surface: 2860 → 2860 rows (dropped 0: drop IV sentinel -99 (inversion failure))
  [3d] surface: 2860 → 2860 rows (dropped 0: drop IV <= 0 (economically invalid))

[3e] Delta nodes present : [np.int64(-80), np.int64(-75), np.int64(-70), np.int64(-65), np.int64(-60), np.int64(-55), np.int64(-50), np.int64(-45), np.int64(-40), np.int64(-35), np.int64(-30), np.int64(-25), np.int64(-20), np.int64(20), np.int64(25), np.int64(30), np.int64(35), np.int64(40), np.int64(45), np.int64(50), np.int64(55), np.int64(60), np.int64(65), np.int64(70), np.int64(75), np.int64(80)]
     Days nodes present   : [np.float64(30.0), np.float64(60.0), np.float64(91.0), np.float64(122.0), np.float64(152.0), np.float64(182.0), np.float64(273.0), np.float64(365.0), np.float64(547.0), np.float64(730.0)]
     Ca

## Step 4 — Compute smile parameters per day

### What is a smile parameter and why do we need it?

The volatility surface gives us IV at many (delta, days) grid points per day. For the regression analysis we need to summarise the *shape* of the smile on each day into a small number of scalar quantities.

We follow the approach standard in the literature (Tompkins 2001, Peña et al. 1999):

**For a given maturity (days) on a given date:**

1. **ATM level** — IV at delta=50 (at-the-money call). This is the overall level of implied volatility.

2. **Skew** — IV(delta=25 put) minus IV(delta=75 call), or equivalently the slope of IV across delta. A more negative skew means the left tail is more expensive (fear of downside). Formally:
   ```
   Skew = put(Δ=25) − call(Δ=75)
Note: this equals −rr in Malz (1997) notation, where rr = call(Δ=0.25) − call(Δ=0.75). Sign convention differs, but instruments are identical.
   ```
   This is the 25-delta risk reversal, widely used in FX and equity vol literature.

3. **Curvature (smile)** — IV(delta=25 put) + IV(delta=75 call) - 2 × IV(delta=50 call). This measures how much the wings deviate from ATM — the 'bend' in the smile. Formally:
   ```
   Curvature = put(Δ=25) + call(Δ=75) − 2×ATM
Note: this equals 2×str in Malz (1997) notation, where str = [call(Δ=0.25) + call(Δ=0.75)]/2 − ATM. Scale differs by a factor of 2, but shape content is identical.
   ```
   This is the 25-delta butterfly spread.

We compute these for each available (date, days) combination.

In [4]:
print("=" * 60)
print("STEP 4 — Compute smile parameters per day")
print("=" * 60)

# ----------------------------------------------------------------
# First: inspect what call/put codes OptionMetrics uses
# EU table uses 'callput' column (vs US 'cp_flag')
# ----------------------------------------------------------------
print("\nCall/put value counts:")
print(surf['callput'].value_counts())

# ----------------------------------------------------------------
# Separate calls and puts
# We will confirm the exact string values from the output above
# before proceeding — DO NOT assume 'C'/'P' without checking
# ----------------------------------------------------------------
callput_values = surf['callput'].unique()
print(f"\nUnique callput values: {callput_values}")
print("\n⚠️  Check the values above before running Step 4b.")
print("    Expected: 'C' and 'P' — if different, update CALL_FLAG and PUT_FLAG below.")

STEP 4 — Compute smile parameters per day

Call/put value counts:
callput
P    1430
C    1430
Name: count, dtype: int64

Unique callput values: ['P' 'C']

⚠️  Check the values above before running Step 4b.
    Expected: 'C' and 'P' — if different, update CALL_FLAG and PUT_FLAG below.


In [6]:
CALL_FLAG = 'C'
PUT_FLAG  = 'P'

calls = surf[surf['callput'] == CALL_FLAG].copy()
puts  = surf[surf['callput'] == PUT_FLAG].copy()

print(f"Calls: {len(calls)} rows")
print(f"Puts : {len(puts)} rows")

# Confirm delta ranges — sanity check before computing
print(f"\nCall delta range: {sorted(calls['delta'].unique())}")
print(f"Put  delta range: {sorted(puts['delta'].unique())}")

smile_records = []

dates     = sorted(surf['date'].unique())
days_list = sorted(surf['days'].unique())

for date in dates:
    for days in days_list:

        c = calls[(calls['date'] == date) & (calls['days'] == days)]
        p = puts[ (puts['date']  == date) & (puts['days']  == days)]

        def get_iv(df, delta):
            """Extract IV at a specific delta node. Cast to int for type safety."""
            row = df[df['delta'] == int(delta)]
            if len(row) == 1:
                return float(row['impliedvol'].values[0])
            return np.nan

        atm_iv = get_iv(c, 50)    # call at delta = +50
        call75 = get_iv(c, 75)    # call at delta = +75
        put25  = get_iv(p, -25)   # put  at delta = -25  ← FIXED (was 25)

        skew      = put25 - call75 \
                    if not (np.isnan(put25) or np.isnan(call75)) else np.nan
        curvature = (put25 + call75 - 2 * atm_iv) \
                    if not (np.isnan(put25) or np.isnan(call75) or np.isnan(atm_iv)) else np.nan

        smile_records.append({
            'date':      date,
            'days':      days,
            'atm_iv':    atm_iv,
            'skew':      skew,
            'curvature': curvature,
            'put25_iv':  put25,
            'call75_iv': call75,
        })

df_smile = pd.DataFrame(smile_records)

print(f"\nSmile parameters computed: {len(df_smile)} (date, days) cells")
print(f"Missing ATM IV   : {df_smile['atm_iv'].isna().sum()}")
print(f"Missing skew     : {df_smile['skew'].isna().sum()}")
print(f"Missing curvature: {df_smile['curvature'].isna().sum()}")
print(f"\nSample (first 10 rows):")
print(df_smile.head(10).to_string(index=False))

Calls: 1430 rows
Puts : 1430 rows

Call delta range: [np.int64(20), np.int64(25), np.int64(30), np.int64(35), np.int64(40), np.int64(45), np.int64(50), np.int64(55), np.int64(60), np.int64(65), np.int64(70), np.int64(75), np.int64(80)]
Put  delta range: [np.int64(-80), np.int64(-75), np.int64(-70), np.int64(-65), np.int64(-60), np.int64(-55), np.int64(-50), np.int64(-45), np.int64(-40), np.int64(-35), np.int64(-30), np.int64(-25), np.int64(-20)]

Smile parameters computed: 110 (date, days) cells
Missing ATM IV   : 0
Missing skew     : 0
Missing curvature: 0

Sample (first 10 rows):
      date  days   atm_iv      skew  curvature  put25_iv  call75_iv
2013-03-01  30.0 0.227476 -0.000448   0.030125  0.242315   0.242763
2013-03-01  60.0 0.226002  0.005566   0.034362  0.245965   0.240400
2013-03-01  91.0 0.226719  0.010123   0.038378  0.250970   0.240847
2013-03-01 122.0 0.228042  0.008866   0.037966  0.251458   0.242592
2013-03-01 152.0 0.229911  0.007803   0.034114  0.250870   0.243067
201

## Step 5 — Clean and align control variables

Each control table needs to be:
1. Date-parsed
2. Filtered to trading days only (using `crsp.dsi` as the calendar)
3. Reduced to the columns we actually need
4. Renamed to clear, unambiguous column names

We build the trading-day calendar first, then align everything to it.

In [10]:
print("=" * 60)
print("STEP 5 — Build trading-day calendar and clean controls")
print("=" * 60)

# ----------------------------------------------------------------
# 5a. Build trading-day calendar from crsp.dsi
# This is the most reliable US equity market calendar
# ----------------------------------------------------------------
cal = df_dsi.copy()
cal['date'] = pd.to_datetime(cal['date'])
trading_days = sorted(cal['date'].unique())
print(f"\n[5a] Trading days in calendar: {len(trading_days)}")
print(f"     From: {min(trading_days).date()} → {max(trading_days).date()}")

# ----------------------------------------------------------------
# 5b. Clean FRB rates — keep only the columns we need
#
# For this thesis we need:
#   dff   = Fed Funds Rate (US risk-free rate proxy, short end)
#   dgs1  = 1-year Treasury yield
#   dgs10 = 10-year Treasury yield
#   t10y2y = 10y-2y yield spread (term structure slope)
#   bamlh0a0hym2 = US HY spread (risk sentiment)
# ----------------------------------------------------------------
frb = df_frb.copy()
frb['date'] = pd.to_datetime(frb['date'])
frb = frb[frb['date'].isin(trading_days)]
frb = frb[['date', 'dff', 'dgs1', 'dgs10', 't10y2y', 'bamlh0a0hym2']].copy()
frb = frb.rename(columns={
    'dff':          'us_rf_rate',
    'dgs1':         'us_yield_1y',
    'dgs10':        'us_yield_10y',
    't10y2y':       'us_term_spread',
    'bamlh0a0hym2': 'us_hy_spread',
})
frb = frb.sort_values('date').reset_index(drop=True)
print(f"\n[5b] FRB rates: {len(frb)} rows, {frb.isna().sum().sum()} total NaNs")
print(f"     Columns: {list(frb.columns)}")

# ----------------------------------------------------------------
# 5c. Clean CRSP market index
#   sprtrn = S&P 500 daily return
#   spindx = S&P 500 index level
# ----------------------------------------------------------------
dsi = df_dsi.copy()
dsi['date'] = pd.to_datetime(dsi['date'])
dsi = dsi[['date', 'sprtrn', 'spindx']].copy()
dsi = dsi.rename(columns={
    'sprtrn': 'sp500_ret',
    'spindx': 'sp500_idx',
})
dsi = dsi.sort_values('date').reset_index(drop=True)
print(f"\n[5c] CRSP DSI: {len(dsi)} rows, {dsi.isna().sum().sum()} total NaNs")

# ----------------------------------------------------------------
# 5d. Clean Fama-French factors
#   mktrf = market excess return
#   rf    = risk-free rate (FF version)
# ----------------------------------------------------------------
ff = df_ff.copy()
ff['date'] = pd.to_datetime(ff['date'])
ff = ff[ff['date'].isin(trading_days)]
ff = ff[['date', 'mktrf', 'smb', 'hml', 'rf']].copy()
ff = ff.rename(columns={
    'mktrf': 'ff_mktrf',
    'smb':   'ff_smb',
    'hml':   'ff_hml',
    'rf':    'ff_rf',
})
ff = ff.sort_values('date').reset_index(drop=True)
print(f"\n[5d] FF factors: {len(ff)} rows, {ff.isna().sum().sum()} total NaNs")

# ----------------------------------------------------------------
# 5e. Clean CBOE VIX
# ----------------------------------------------------------------
vix = df_vix.copy()
vix['date'] = pd.to_datetime(vix['date'])
vix = vix[vix['date'].isin(trading_days)]
vix = vix[['date', 'vix']].copy()
vix = vix.sort_values('date').reset_index(drop=True)
print(f"\n[5e] VIX: {len(vix)} rows, {vix.isna().sum().sum()} total NaNs")

# ----------------------------------------------------------------
# 5f. Construct EUR/USD from GBP cross rates
#
# The table only has GBP as the base currency (fromcurd == 'GBP').
# Every row answers: "how many units of tocurd does 1 GBP buy?"
#
# So:
#   GBP/USD = exratd where tocurd == 'USD'
#   GBP/EUR = exratd where tocurd == 'EUR'
#
# EUR/USD = GBP/USD / GBP/EUR
#
# This is triangular arbitrage arithmetic — no assumption beyond
# the definition of an exchange rate.
# ----------------------------------------------------------------

fx_raw = df_fx.copy()
fx_raw['date'] = pd.to_datetime(fx_raw['datadate'])

# Keep only AR (actual rate) rows — not CF (conversion factor)
fx_raw = fx_raw[fx_raw['exrattpd'] == 'AR']

# Extract GBP/USD
gbp_usd = fx_raw[fx_raw['tocurd'] == 'USD'][['date', 'exratd']].copy()
gbp_usd = gbp_usd.rename(columns={'exratd': 'gbp_usd'})

# Extract GBP/EUR
gbp_eur = fx_raw[fx_raw['tocurd'] == 'EUR'][['date', 'exratd']].copy()
gbp_eur = gbp_eur.rename(columns={'exratd': 'gbp_eur'})

# Merge on date
fx_cross = gbp_usd.merge(gbp_eur, on='date', how='inner')

# Compute EUR/USD = GBP/USD / GBP/EUR
fx_cross['eurusd'] = fx_cross['gbp_usd'] / fx_cross['gbp_eur']

# Filter to trading days only
fx_cross = fx_cross[fx_cross['date'].isin(trading_days)]

# Keep only what we need
fx = fx_cross[['date', 'eurusd']].copy()
fx = fx.sort_values('date').reset_index(drop=True)

print(f"[5f] EUR/USD: {len(fx)} rows, {fx.isna().sum().sum()} total NaNs")
print(f"     Date range: {fx['date'].min().date()} → {fx['date'].max().date()}")
print(f"     Sample values (first 5):")
print(fx.head().to_string(index=False))
print(f"     EUR/USD mean: {fx['eurusd'].mean():.4f}")
print(f"     EUR/USD range: {fx['eurusd'].min():.4f} → {fx['eurusd'].max():.4f}")

# ----------------------------------------------------------------
# 5g. Clean historical volatility (realized vol)
# Keep only our security, select a standard horizon (30 days)
# ----------------------------------------------------------------
hv = df_histvol.copy()
hv['date'] = pd.to_datetime(hv['date'])
hv = hv[hv['securityid'] == EU_SECURITYID]
print(f"\n[5g] Historical vol — available days horizons: {sorted(hv['days'].unique())}")

# Keep 30-day realized vol as our benchmark
# If 30 is not available we take the nearest available
if 30 in hv['days'].values:
    hv30 = hv[hv['days'] == 30][['date', 'volatility']].copy()
else:
    nearest = hv['days'].unique()[np.argmin(np.abs(hv['days'].unique() - 30))]
    print(f"     30-day HV not found — using {nearest}-day instead")
    hv30 = hv[hv['days'] == nearest][['date', 'volatility']].copy()

hv30 = hv30.rename(columns={'volatility': 'hist_vol_30d'})
hv30 = hv30.sort_values('date').reset_index(drop=True)
print(f"     HV30 rows: {len(hv30)}, NaNs: {hv30.isna().sum().sum()}")

print("\n✓ Step 5 complete.")

STEP 5 — Build trading-day calendar and clean controls

[5a] Trading days in calendar: 82
     From: 2013-01-02 → 2013-04-30

[5b] FRB rates: 82 rows, 0 total NaNs
     Columns: ['date', 'us_rf_rate', 'us_yield_1y', 'us_yield_10y', 'us_term_spread', 'us_hy_spread']

[5c] CRSP DSI: 82 rows, 0 total NaNs

[5d] FF factors: 82 rows, 0 total NaNs

[5e] VIX: 82 rows, 0 total NaNs
[5f] EUR/USD: 82 rows, 0 total NaNs
     Date range: 2013-01-02 → 2013-04-30
     Sample values (first 5):
      date   eurusd
2013-01-02 1.324010
2013-01-03 1.309375
2013-01-04 1.304457
2013-01-07 1.309691
2013-01-08 1.306664
     EUR/USD mean: 1.3155
     EUR/USD range: 1.2781 → 1.3695

[5g] Historical vol — available days horizons: [np.float64(10.0), np.float64(14.0), np.float64(30.0), np.float64(60.0), np.float64(91.0), np.float64(122.0), np.float64(152.0), np.float64(182.0), np.float64(273.0), np.float64(365.0), np.float64(547.0), np.float64(730.0), np.float64(1825.0)]
     HV30 rows: 11, NaNs: 0

✓ Step 5 comp

In [8]:
# ── EUR/USD diagnostic can be ignored
print("comp.g_exrt_dly — raw inspection")
print(f"\nShape: {df_fx.shape}")
print(f"\nColumn names:\n{df_fx.columns.tolist()}")
print(f"\nFirst 5 rows:\n{df_fx.head().to_string()}")
print(f"\nUnique values in each column:")
for col in df_fx.columns:
    uvals = df_fx[col].unique()
    print(f"  {col}: {uvals[:10]}")  # show first 10 unique values per column

comp.g_exrt_dly — raw inspection

Shape: (20719, 5)

Column names:
['tocurd', 'exratd', 'exrattpd', 'fromcurd', 'datadate']

First 5 rows:
  tocurd   exratd exrattpd fromcurd    datadate
0    GBP   1.0000       AR      GBP  2013-01-01
1    EGP  10.3223       AR      GBP  2013-01-01
2    AUD   1.5654       AR      GBP  2013-01-01
3    BMD   1.6252       AR      GBP  2013-01-01
4    CAD   1.6183       AR      GBP  2013-01-01

Unique values in each column:
  tocurd: ['GBP' 'EGP' 'AUD' 'BMD' 'CAD' 'FJD' 'GYD' 'HKD' 'JMD' 'KYD']
  exratd: [  1.      10.3223   1.5654   1.6252   1.6183   2.8809 331.39    12.598
 150.09     1.3247]
  exrattpd: ['AR' 'CF']
  fromcurd: ['GBP']
  datadate: ['2013-01-01' '2013-01-02' '2013-01-03' '2013-01-04' '2013-01-05'
 '2013-01-06' '2013-01-07' '2013-01-08' '2013-01-09' '2013-01-10']


In [9]:
print("tocurd unique values:")
print(sorted(df_fx['tocurd'].unique()))

# Return to step 5f 

tocurd unique values:
['AED', 'AFN', 'ALL', 'AMD', 'ANG', 'AOA', 'ARS', 'ATS', 'AUD', 'AWG', 'AZM', 'AZN', 'BAM', 'BBD', 'BDT', 'BEF', 'BGN', 'BHD', 'BIF', 'BMD', 'BND', 'BOB', 'BRL', 'BSD', 'BTN', 'BWP', 'BYR', 'BZD', 'CAD', 'CDF', 'CHF', 'CLF', 'CLP', 'CNH', 'CNY', 'COP', 'CRC', 'CUP', 'CVE', 'CYP', 'CZK', 'DEM', 'DJF', 'DKK', 'DOP', 'DZD', 'ECS', 'EEK', 'EGP', 'ESP', 'ETB', 'EUR', 'FIM', 'FJD', 'FRF', 'GBP', 'GEL', 'GHS', 'GMD', 'GNF', 'GRD', 'GTQ', 'GYD', 'HKD', 'HNL', 'HRK', 'HTG', 'HUF', 'IDR', 'IEP', 'ILS', 'INR', 'IQD', 'IRR', 'ISK', 'ITL', 'JMD', 'JOD', 'JPY', 'KES', 'KGS', 'KHR', 'KMF', 'KPW', 'KRW', 'KWD', 'KYD', 'KZT', 'LAK', 'LBP', 'LKR', 'LRD', 'LSL', 'LTL', 'LUF', 'LVL', 'LYD', 'MAD', 'MDL', 'MGA', 'MKD', 'MMK', 'MNT', 'MOP', 'MRO', 'MTL', 'MUR', 'MVR', 'MWK', 'MXN', 'MYR', 'MZN', 'NAD', 'NGN', 'NIO', 'NLG', 'NOK', 'NPR', 'NZD', 'OMR', 'PAB', 'PEN', 'PGK', 'PHP', 'PKR', 'PLN', 'PTE', 'PYG', 'QAR', 'RON', 'RSD', 'RUB', 'RWF', 'SAR', 'SBD', 'SCR', 'SDG', 'SEK', 'SGD', 'SIT

## Step 6 — Merge into one daily panel

### Merge strategy

The base of the merge is the **smile parameter table** (`df_smile`), which has one row per (date, days) cell — the natural unit of analysis for smile regressions.

All control variables are merged on `date` using a **left join** from the smile table. This means:
- Every (date, days) row in the smile table is kept
- Control values are attached where available
- Missing controls on a given day produce NaN (not dropped silently)

After the merge we log the shape and missingness of the final table.

In [11]:
print("=" * 60)
print("STEP 6 — Merge into daily panel")
print("=" * 60)

panel = df_smile.copy()
n_base = len(panel)
print(f"\nBase (smile): {n_base} rows")

# Merge controls one by one — left join on date
merges = [
    (frb,  'date', 'FRB rates'),
    (dsi,  'date', 'CRSP S&P500'),
    (ff,   'date', 'FF factors'),
    (vix,  'date', 'VIX'),
    (fx,   'date', 'EUR/USD'),
    (hv30, 'date', 'Hist vol 30d'),
]

for df_ctrl, key, label in merges:
    n_before = len(panel)
    panel = panel.merge(df_ctrl, on=key, how='left')
    n_after = len(panel)
    if n_before != n_after:
        print(f"  ⚠️  WARNING: row count changed after merging {label}: {n_before} → {n_after}")
    else:
        print(f"  ✓ Merged {label}: {n_after} rows")

panel = panel.sort_values(['date', 'days']).reset_index(drop=True)

print(f"\nFinal panel shape: {panel.shape}")
print(f"Columns: {list(panel.columns)}")

STEP 6 — Merge into daily panel

Base (smile): 110 rows
  ✓ Merged FRB rates: 110 rows
  ✓ Merged CRSP S&P500: 110 rows
  ✓ Merged FF factors: 110 rows
  ✓ Merged VIX: 110 rows
  ✓ Merged EUR/USD: 110 rows
  ✓ Merged Hist vol 30d: 110 rows

Final panel shape: (110, 21)
Columns: ['date', 'days', 'atm_iv', 'skew', 'curvature', 'put25_iv', 'call75_iv', 'us_rf_rate', 'us_yield_1y', 'us_yield_10y', 'us_term_spread', 'us_hy_spread', 'sp500_ret', 'sp500_idx', 'ff_mktrf', 'ff_smb', 'ff_hml', 'ff_rf', 'vix', 'eurusd', 'hist_vol_30d']


## Step 7 — Final validation of merged dataset

In [12]:
print("=" * 60)
print("STEP 7 — Final validation")
print("=" * 60)

print(f"\n[1] Shape: {panel.shape[0]:,} rows x {panel.shape[1]} columns")

print(f"\n[2] Date coverage:")
print(f"    {panel['date'].min().date()} → {panel['date'].max().date()}")
print(f"    Unique trading days: {panel['date'].nunique()}")
print(f"    Unique maturity nodes (days): {sorted(panel['days'].unique())}")

print(f"\n[3] Missingness by column:")
miss = panel.isna().sum()
miss = miss[miss > 0]
if len(miss) == 0:
    print("    No missing values.")
else:
    for col, n in miss.items():
        print(f"    {col:<25} {n:>5} missing ({100*n/len(panel):.1f}%)")

print(f"\n[4] Duplicate rows: {panel.duplicated().sum()}")

print(f"\n[5] Key variable summary stats:")
key_cols = ['atm_iv', 'skew', 'curvature', 'us_rf_rate', 'vix', 'eurusd', 'sp500_ret']
available = [c for c in key_cols if c in panel.columns]
print(panel[available].describe().round(4).to_string())

print(f"\n[6] Transformation log summary:")
for entry in transform_log:
    print(f"    [{entry['step']}] {entry['table']}: dropped {entry['dropped']} rows — {entry['reason']}")

STEP 7 — Final validation

[1] Shape: 110 rows x 21 columns

[2] Date coverage:
    2013-03-01 → 2013-03-15
    Unique trading days: 11
    Unique maturity nodes (days): [np.float64(30.0), np.float64(60.0), np.float64(91.0), np.float64(122.0), np.float64(152.0), np.float64(182.0), np.float64(273.0), np.float64(365.0), np.float64(547.0), np.float64(730.0)]

[3] Missingness by column:
    No missing values.

[4] Duplicate rows: 0

[5] Key variable summary stats:
         atm_iv      skew  curvature  us_rf_rate       vix    eurusd  sp500_ret
count  110.0000  110.0000   110.0000    110.0000  110.0000  110.0000   110.0000
mean     0.2177    0.0142     0.0356      0.1518   12.7536    1.3010     0.0027
std      0.0103    0.0075     0.0055      0.0072    1.2251    0.0037     0.0032
min      0.1942   -0.0017     0.0107      0.1400   11.3000    1.2950    -0.0024
25%      0.2106    0.0079     0.0340      0.1500   11.5600    1.2983     0.0011
50%      0.2164    0.0185     0.0364      0.1500   12.5

## Step 8 — Save to intermediate

In [13]:
print("=" * 60)
print("STEP 8 — Save")
print("=" * 60)

# Save analysis-ready panel
out_path = os.path.join(INTERMEDIATE, f"eu2013_analysis_ready__{TIMESTAMP}.csv")
panel.to_csv(out_path, index=False)
print(f"\n✓ Panel saved to: {out_path}")
print(f"  Shape: {panel.shape[0]:,} rows x {panel.shape[1]} columns")

# Save transformation log
log_path = os.path.join(LOG_DIR, f"eu2013_pipeline_log__{TIMESTAMP}.json")
with open(log_path, 'w') as f:
    json.dump(transform_log, f, indent=2, default=str)
print(f"✓ Transform log saved to: {log_path}")

print("\n✓ Notebook 03a complete.")
print("\nNext steps:")
print("  - Inspect the panel and the smile parameters")
print("  - Run notebook 03b (US 2014 pipeline) with the same structure")
print("  - With full OptionMetrics access: re-run on full SPX/OESX data")

STEP 8 — Save

✓ Panel saved to: ../data/intermediate/eu2013_analysis_ready__20260312_113334.csv
  Shape: 110 rows x 21 columns
✓ Transform log saved to: ../logs/eu2013_pipeline_log__20260312_113334.json

✓ Notebook 03a complete.

Next steps:
  - Inspect the panel and the smile parameters
  - Run notebook 03b (US 2014 pipeline) with the same structure
  - With full OptionMetrics access: re-run on full SPX/OESX data
